<a href="https://colab.research.google.com/github/AbeChessell/AbeChessell-Programming-BigDataAnalytics/blob/main/3_02_key_value_storesCOMPLETED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1vv_PsWBnUJwSCkwKDoJAC-vXjtaEA4Ts)

# 3.02 Key-value Stores with TinyDB
This tutorial gives a basic introduction to working with key-value (KV) stores (or document DBs). We will be working with [TinyDB](https://tinydb.readthedocs.io/en/latest/index.html), an in-memory Python database, which is particularly attractive here as it is, as the name suggests, pretty small and lightweight.

We will begin with the relevant installs:

In [ ]:
!pip install tinydb
!pip install faker
!pip install python-lorem

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for python-lorem: filename=python_lorem-1.3.0.post3-py3-none-any.whl size=9166 sha256=fc6f9c6cdde8a6d3e9311cb4e38e566372af49481359fd85b2f35b75455bcc38
  Stored in directory: /root/.cache/pip/wheels/9d/3a/d2/ba74831189552ab46d4b3d52d3ef86e4f06feef77b0a35d0ea
Successfully built python-lorem


As you may infer from the pacakages installed, we will run something similar to one of our DuckDB examples - specifically building a database using fake data generated by Faker:

In [ ]:
import random
from faker import Faker
import pandas as pd
from lorem import paragraph
import itertools

fake = Faker()

def get_person():
  person = {}
  person['id'] = random.randrange(1000,9999999999999)
  person['first_name'] = fake.first_name()
  person['last_name'] = fake.last_name()
  person['email'] = fake.unique.ascii_email()
  person['company'] = fake.company()
  person['phone'] = fake.phone_number()
  person['review'] = list(itertools.islice(paragraph(count=1), 1))
  return person

personlist = []
for x in range(100):
  personlist.append(get_person())

df = pd.DataFrame.from_dict(personlist)
df.head()

,id,first_name,last_name,email,company,phone,review
0,3000264651499,James,Acevedo,tiffanymeyer@garrison.com,"Morales, Lopez and Garcia",755.300.4935x921,[Ad in elit et. Aute commodo velit exercitatio...
1,1689247237777,Brittney,Jones,cbailey@gmail.com,Roth-Clark,(526)258-3724,[Aliqua eu veniam aute id. Officia qui et cons...
2,9251826501762,Veronica,Berry,rortiz@benson.com,Collier Ltd,217-214-1586x800,[Est cupidatat fugiat duis voluptate dolor exc...
3,2799974428386,Phillip,Steele,cindy94@wright-cobb.com,Irwin and Sons,(460)650-3900,[Duis ipsum occaecat et ea incididunt voluptat...
4,7420080423418,Lori,Adams,jonescalvin@hotmail.com,Rogers Group,9622499551,[Proident occaecat cillum cillum eiusmod nisi ...


Everything here is the same except we have also add a text column (using lorem ipsum). As before we have created this as a Pandas dataframe, but like most KV stores, TinyDB prefers data stored as a dictionary:

In [ ]:
fake_data = df.to_dict(orient='records')
fake_data

[{'id': 3000264651499,
  'first_name': 'James',
  'last_name': 'Acevedo',
  'email': 'tiffanymeyer@garrison.com',
  'company': 'Morales, Lopez and Garcia',
  'phone': '755.300.4935x921',
  'review': ['Ad in elit et. Aute commodo velit exercitation amet amet. Sunt sed culpa duis fugiat eu duis elit, culpa proident magna cupidatat cupidatat eiusmod mollit sunt, incididunt cillum pariatur aliqua ipsum id do esse. Ex mollit irure fugiat ex lorem. Eu enim consequat lorem consectetur duis. Aliqua fugiat aute id ex quis ut nostrud. Dolore non sunt ex deserunt fugiat ex. Non sed ullamco veniam esse, enim non deserunt ullamco.']},
 {'id': 1689247237777,
  'first_name': 'Brittney',
  'last_name': 'Jones',
  'email': 'cbailey@gmail.com',
  'company': 'Roth-Clark',
  'phone': '(526)258-3724',
  'review': ['Aliqua eu veniam aute id. Officia qui et consectetur anim eu dolor. Eiusmod magna consequat nostrud ea, enim excepteur qui minim. Officia ullamco amet fugiat fugiat magna qui non, laboris do eu 

With this transform in place we can load the data into our database. You may note the database itself is specified as JSON format:

In [ ]:
from tinydb import TinyDB, Query

db = TinyDB('db.json')

for record in fake_data:
  db.insert(record)

We can check this has worked with a simple Python loop:

In [ ]:
for item in db:
  print(item)

{'id': 3000264651499, 'first_name': 'James', 'last_name': 'Acevedo', 'email': 'tiffanymeyer@garrison.com', 'company': 'Morales, Lopez and Garcia', 'phone': '755.300.4935x921', 'review': ['Ad in elit et. Aute commodo velit exercitation amet amet. Sunt sed culpa duis fugiat eu duis elit, culpa proident magna cupidatat cupidatat eiusmod mollit sunt, incididunt cillum pariatur aliqua ipsum id do esse. Ex mollit irure fugiat ex lorem. Eu enim consequat lorem consectetur duis. Aliqua fugiat aute id ex quis ut nostrud. Dolore non sunt ex deserunt fugiat ex. Non sed ullamco veniam esse, enim non deserunt ullamco.']}
{'id': 1689247237777, 'first_name': 'Brittney', 'last_name': 'Jones', 'email': 'cbailey@gmail.com', 'company': 'Roth-Clark', 'phone': '(526)258-3724', 'review': ['Aliqua eu veniam aute id. Officia qui et consectetur anim eu dolor. Eiusmod magna consequat nostrud ea, enim excepteur qui minim. Officia ullamco amet fugiat fugiat magna qui non, laboris do eu anim velit. Labore elit eli

With our database setup, we can start to query our records. In TinyDB we do this by creating a query object:

In [ ]:
User = Query() # query object

db.search(User.first_name == 'Chad') # adapt based on your data

[{'id': 8289021419344,
  'first_name': 'Chad',
  'last_name': 'Johnson',
  'email': 'alex87@gmail.com',
  'company': 'Cochran, Hayes and Williams',
  'phone': '(805)811-1629x9255',
  'review': ['Et labore aute minim irure ullamco pariatur qui. Dolor excepteur in veniam ea ipsum et irure, eu commodo laborum nulla velit dolor in eu, elit dolore dolor exercitation cupidatat occaecat. Deserunt lorem duis mollit aute dolor, esse cillum ut elit. Voluptate fugiat laborum cupidatat sunt est, quis esse ipsum nisi irure et proident labore. Sint irure elit do incididunt id labore. Aute aliqua aliqua veniam consectetur.']}]

We can also add new data in dictionary/JSON-like format:

In [ ]:
db.insert({'id': 123, 'first_name': 'Amir', 'star_sign': 'Dog', 'review': 'I do not speak Latin.'})

101

And retrieve the data as before:

In [ ]:
db.search(User.id == 123)

[{'id': 123,
  'first_name': 'Amir',
  'star_sign': 'Dog',
  'review': 'I do not speak Latin.'}]

One thing to note here is that our new record does not follow the schema we may infer from the original dataset (i.e. the original data all used the same columns/fields). Here many of those fields are missing and we have the new field 'star_sign'.

This demonstrates the extra flexibility we get with a KV store over a relational model. We can also query our database to get all records that have a specific field:

In [ ]:
db.search(User.star_sign.exists())

[{'id': 123,
  'first_name': 'Amir',
  'star_sign': 'Dog',
  'review': 'I do not speak Latin.'}]

This gives a basic intro into KV (and document) stores. While there are many competing brands/solutions, the common themes are the dictionary-like structure (key-value pairs) and flexibility to accept any fields (keys).